# Mixed-level training: Low + 10% of each class from Medium and High
Exploratory extension, not the paper's Low-only scenarios. Seed 42; original timing trio and window=3 unchanged; each expert has 40 trees; reference RF has 100 trees.

For each application capture in Medium/High, the first floor(0.10 × N) chronologically sorted retained records enter training. All Low records also enter training. The remaining 90% are candidate tests. Remove candidate test records whose bidirectional TCP five-tuple occurs anywhere in the mixed training set. This conservative exclusion also removes reused tuples across captures. No feature values or test outcomes influence selection. Train and test windows are computed separately. Warm-up rows are dropped, so model sample counts differ from raw split counts.

Fit a Low-only control and mixed-level model; evaluate both on the exact same purged Medium and High tests. RF comparisons use identical rows and training inputs (with a separately identified raw-timing reference). Using all Low plus prefixes of other captures is within-capture adaptation, not generalization to independent captures. No claim that unsupervised clusters correspond to true Low/Medium/High is made. Other congestion levels in training were already inspected in previous experiments; results are exploratory, not a sealed final test.


In [ ]:
from pathlib import Path
from collections import deque
import hashlib, json, platform, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, silhouette_score)
from IPython.display import display

SEED=42
WINDOW=3
TIMING=['TcpRtt','SynAck','AckDat']
STATS=['mean','max','median','min','std']
CLASSES=['HTTP','SFTP','SMTP','SSH','Video']
MBK_CONFIG=dict(n_clusters=3,batch_size=1024,n_init=10,max_iter=100,random_state=SEED)
RF_CONFIG=dict(n_estimators=40,criterion='gini',max_depth=None,min_samples_leaf=1,
               max_features='sqrt',bootstrap=True,class_weight=None,n_jobs=-1,random_state=SEED)
RUN_REFERENCE_RF=True
DATA_DIR_OVERRIDE=None
bases=[Path.cwd(),*Path.cwd().parents]
candidates=[b/r for b in bases for r in ['CDR_MLC/DATASETS/CDR-MLC/New_Version','DATASETS/CDR-MLC/New_Version']]
DATA_DIR=Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else next((p for p in candidates if p.is_dir()),None)
if DATA_DIR is None: raise FileNotFoundError('Set DATA_DIR_OVERRIDE')
OUT=DATA_DIR.parents[2]/'outputs'/'mixed_training_10pct'
OUT.mkdir(parents=True,exist_ok=True)
print('Data:',DATA_DIR,'\nOutput:',OUT)
print('Versions:',platform.python_version(),pd.__version__,np.__version__,sklearn.__version__)


In [ ]:
SERVICES={'HTTP':('192.168.2.122',8080),'SFTP':('192.168.2.120',22),
          'SMTP':('192.168.2.120',8025),'SSH':('192.168.2.120',22),'Video':('192.168.2.121',5000)}
CLIENT='192.168.1.111'
frames=[];audit=[]
for p in sorted(DATA_DIR.glob('*.flow')):
    m=re.fullmatch(r'(HTTP|SFTP|SMTP|SSH|Video)_(Low|Medium|High)',p.stem)
    if not m: raise ValueError(f'Unexpected capture: {p.name}')
    label,level=m.groups(); server,port=SERVICES[label]
    d=pd.read_csv(p,low_memory=False,on_bad_lines='error')
    d.columns=d.columns.str.strip()
    required=['StartTime','SrcAddr','DstAddr','Proto','Sport','Dport',*TIMING]
    if set(required)-set(d):raise ValueError(f'Missing columns in {p.name}')
    for c in d.select_dtypes('object'):d[c]=d[c].str.strip().replace('',np.nan)
    d['source_row']=np.arange(len(d))+2
    tcp=d.Proto.eq('tcp')
    forward=tcp & d.SrcAddr.eq(CLIENT) & d.DstAddr.eq(server) & pd.to_numeric(d.Dport,errors='coerce').eq(port)
    reverse=tcp & d.SrcAddr.eq(server) & d.DstAddr.eq(CLIENT) & pd.to_numeric(d.Sport,errors='coerce').eq(port)
    if reverse.any():raise ValueError(f'{p.name}: reverse-oriented records require canonicalization before analysis')
    audit.append(dict(file=p.name,raw_records=len(d),retained=int(forward.sum()),excluded=int((~forward).sum()),
                      sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
    d=d.loc[forward].copy()
    if d.empty:raise ValueError(f'No service records in {p.name}')
    d['timestamp']=pd.to_datetime(d.StartTime,format='%Y/%m/%d %H:%M:%S.%f',errors='raise')
    for c in TIMING:d[c]=pd.to_numeric(d[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    d['traffic_label']=label;d['congestion_level']=level;d['sequence_id']=p.stem;d['source_file']=p.name
    frames.append(d.sort_values(['timestamp','source_row'],kind='stable'))
data=pd.concat(frames,ignore_index=True)
assert set(zip(data.traffic_label,data.congestion_level))=={(c,l) for c in CLASSES for l in ['Low','Medium','High']}
aud=pd.DataFrame(audit);display(aud);aud.to_csv(OUT/'input_audit.csv',index=False)
display(data.groupby(['traffic_label','congestion_level']).size().unstack().reindex(columns=['Low','Medium','High']))


In [ ]:
TREND_COLUMNS=[f'{f}_{s}' for f in TIMING for s in STATS]
def trend_windows(frame):
    chunks=[];report=[]
    for sequence,g in frame.groupby('sequence_id',sort=False):
        g=g.sort_values(['timestamp','source_row'],kind='stable').copy()
        valid=np.isfinite(g[TIMING]).all(axis=1)&g[TIMING].ge(0).all(axis=1)
        segment=(~valid).cumsum()
        pieces=[]
        for _,part in g.loc[valid].groupby(segment[valid],sort=False):
            z=part.copy()
            for f in TIMING:
                roll=part[f].rolling(WINDOW,min_periods=WINDOW)
                for stat in STATS:z[f'{f}_{stat}']=roll.std(ddof=0) if stat=='std' else getattr(roll,stat)()
            pieces.append(z.dropna(subset=TREND_COLUMNS))
        kept=pd.concat(pieces) if pieces else g.iloc[:0].assign(**{c:np.nan for c in TREND_COLUMNS})
        report.append(dict(sequence_id=sequence,input_records=len(g),invalid_timing=int((~valid).sum()),
                           warmup=int(valid.sum()-len(kept)),usable=len(kept)))
        chunks.append(kept)
    return pd.concat(chunks,ignore_index=True),pd.DataFrame(report)

class StreamTrend:
    def __init__(self):self.buffers={}
    def update(self,sequence_id,values):
        a=np.asarray(values,dtype=float)
        q=self.buffers.setdefault(sequence_id,deque(maxlen=WINDOW))
        if not np.isfinite(a).all() or (a<0).any():q.clear();return None
        q.append(a)
        if len(q)<WINDOW:return None
        x=np.asarray(q)
        return np.array([v for j in range(3) for v in (x[:,j].mean(),x[:,j].max(),np.median(x[:,j]),x[:,j].min(),x[:,j].std(ddof=0))])

# Verify ordering, warm-up and the exact vector used by inference.
probe=data.groupby('sequence_id',sort=False).head(20).copy()
batch,_=trend_windows(probe)
s=StreamTrend(); stream=[]
for _,r in probe.iterrows():
    v=s.update(r.sequence_id,r[TIMING].to_numpy(float))
    if v is not None:stream.append(v)
assert np.allclose(np.asarray(stream),batch[TREND_COLUMNS].to_numpy(),atol=1e-10)
print('Streaming/batch window equivalence passed.')


In [ ]:
META={'StartTime','SrcAddr','DstAddr','Proto','Sport','Dport','Label','Cause','Dir','sTtl','dTtl',
      'traffic_label','congestion_level','sequence_id','source_file','source_row','timestamp'}
CATEGORICAL=['Flgs','State','TcpOpt']
def select_columns(train):
    nums=[];cats=[];reasons=[]
    for c in train.columns:
        reason=None
        if c in META or c in TREND_COLUMNS:reason='metadata_or_routing_statistics'
        elif re.search(r'\.\d+$',c):reason='duplicate_header'
        elif c=='IdleTime':reason='unresolved_semantics'
        elif c in CATEGORICAL:
            if train[c].nunique(dropna=True)>1:cats.append(c)
            else:reason='empty_or_constant'
        else:
            n=pd.to_numeric(train[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
            if not n.notna().any():reason='empty_or_non_numeric'
            elif n.nunique()<2:reason='constant'
            elif any(n.equals(pd.to_numeric(train[k],errors='coerce')) for k in nums):reason='identical_numeric_column'
            else:nums.append(c)
        if reason:reasons.append({'feature':c,'reason':reason})
    return nums,cats,pd.DataFrame(reasons)
def normalize_inputs(frame,nums,cats):
    d=frame[nums+cats].copy()
    for c in nums:d[c]=pd.to_numeric(d[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in cats:d[c]=d[c].fillna('__MISSING__').astype(str)
    return d

def fit_cdr(train_raw):
    train,window_report=trend_windows(train_raw)
    if len(train)<3:raise ValueError('Insufficient training windows')
    scaler=StandardScaler().fit(train[TREND_COLUMNS])
    z=scaler.transform(train[TREND_COLUMNS])
    router=MiniBatchKMeans(**MBK_CONFIG).fit(z)
    routes=router.predict(z)
    if len(np.unique(routes))!=3:raise ValueError('Training failed to produce three nonempty regimes')
    nums,cats,reasons=select_columns(train)
    expert_nums=[c for c in nums if c not in TIMING]
    assert not (set(expert_nums+cats)&(set(TIMING)|META|set(TREND_COLUMNS)))
    transformers=[]
    if expert_nums:transformers.append(('numeric',SimpleImputer(strategy='median'),expert_nums))
    if cats:transformers.append(('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats))
    pre=ColumnTransformer(transformers,remainder='drop')
    x=pre.fit_transform(normalize_inputs(train,expert_nums,cats))
    experts={}
    for k in range(3):
        mask=routes==k
        experts[k]=RandomForestClassifier(**RF_CONFIG).fit(x[mask],train.loc[mask,'traffic_label'])
    # Cluster numbers remain arbitrary: no mapping to true congestion labels for inference.
    cluster_classes=pd.crosstab(pd.Series(routes,name='cluster'),train.traffic_label).reindex(columns=CLASSES,fill_value=0)
    print('Training cluster/class counts:');display(cluster_classes)
    if (cluster_classes==0).any().any():print('Some experts lack training classes; no oracle fallback is used.')
    print('Expert raw numeric/categorical fields:',len(expert_nums),len(cats),'encoded columns:',x.shape[1])
    sample=min(3000,len(z))
    try: silhouette=silhouette_score(z,routes,sample_size=sample,random_state=SEED)
    except ValueError:silhouette=np.nan
    return dict(router=router,scaler=scaler,pre=pre,experts=experts,nums=expert_nums,cats=cats,
                train=train,routes=routes,window_report=window_report,reasons=reasons,
                cluster_classes=cluster_classes,silhouette=silhouette,encoded_features=pre.get_feature_names_out().tolist())

def predict_cdr(model,test_raw):
    test,report=trend_windows(test_raw)
    routes=model['router'].predict(model['scaler'].transform(test[TREND_COLUMNS]))
    x=model['pre'].transform(normalize_inputs(test,model['nums'],model['cats']))
    pred=np.empty(len(test),dtype=object)
    for k,expert in model['experts'].items():
        mask=routes==k
        if mask.any():pred[mask]=expert.predict(x[mask])
    return test,pred,routes,report

def predict_record(model,stream,sequence_id,record):
    # record contains observed features only; does not read target or congestion labels.
    trend=stream.update(sequence_id,[record[f] for f in TIMING])
    if trend is None:return None
    z=pd.DataFrame([trend],columns=TREND_COLUMNS)
    k=int(model['router'].predict(model['scaler'].transform(z))[0])
    raw=pd.DataFrame([record])
    x=model['pre'].transform(normalize_inputs(raw,model['nums'],model['cats']))
    return str(model['experts'][k].predict(x)[0]),k


In [ ]:
def evaluate(y,pred,name,folder):
    p,r,f,_=precision_recall_fscore_support(y,pred,average='weighted',zero_division=0)
    macro=precision_recall_fscore_support(y,pred,average='macro',zero_division=0)[2]
    metrics=dict(method=name,accuracy=accuracy_score(y,pred),precision_weighted=p,recall_weighted=r,f1_weighted=f,f1_macro=macro)
    report=classification_report(y,pred,labels=CLASSES,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(folder/f'{name}_classification_report.csv')
    cm=confusion_matrix(y,pred,labels=CLASSES)
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(folder/f'{name}_confusion.csv')
    return metrics

def run_mixed(name,tr,te,model=None):
    folder=OUT/name;folder.mkdir(parents=True,exist_ok=True)
    assert set(zip(tr.source_file,tr.source_row)).isdisjoint(set(zip(te.source_file,te.source_row)))
    train_level='+'.join(sorted(tr.congestion_level.unique()))
    test_level='+'.join(sorted(te.congestion_level.unique()))
    model=fit_cdr(tr) if model is None else model
    test,pred,routes,test_windows=predict_cdr(model,te)
    # Verify inference does not depend on either target or congestion annotation.
    altered=te.copy();altered['traffic_label']='HIDDEN';altered['congestion_level']='HIDDEN'
    _,pred2,routes2,_=predict_cdr(model,altered)
    assert np.array_equal(pred,pred2) and np.array_equal(routes,routes2)
    # Compare streaming and batch predictions on the first few raw rows of every sequence.
    small=te.groupby('sequence_id',sort=False).head(8)
    bt,bp,br,_=predict_cdr(model,small)
    state=StreamTrend();sp=[];sr=[]
    for _,record in small.iterrows():
        result=predict_record(model,state,record.sequence_id,record.to_dict())
        if result is not None:sp.append(result[0]);sr.append(result[1])
    assert list(bp)==sp and list(br)==sr
    rows=[evaluate(test.traffic_label,pred,'CDR_MLC',folder)]
    if RUN_REFERENCE_RF:
        for include_timing in [False,True]:
            nums=model['nums']+(TIMING if include_timing else [])
            cats=model['cats'];trans=[('numeric',SimpleImputer(strategy='median'),nums)]
            if cats:trans.append(('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats))
            baseline=Pipeline([('pre',ColumnTransformer(trans)),('rf',RandomForestClassifier(**{**RF_CONFIG,'n_estimators':100}))])
            baseline.fit(normalize_inputs(model['train'],nums,cats),model['train'].traffic_label)
            bp=baseline.predict(normalize_inputs(test,nums,cats))
            rows.append(evaluate(test.traffic_label,bp,'RF_100_with_timing' if include_timing else 'RF_100_expert_inputs',folder))
    metrics=pd.DataFrame(rows).set_index('method');display(metrics)
    model['window_report'].to_csv(folder/'train_windows.csv',index=False)
    test_windows.to_csv(folder/'test_windows.csv',index=False)
    model['reasons'].to_csv(folder/'excluded_features.csv',index=False)
    model['cluster_classes'].to_csv(folder/'train_cluster_classes.csv')
    metrics.to_csv(folder/'metrics.csv')
    predictions=test[['source_file','source_row','timestamp','traffic_label','congestion_level']].copy()
    predictions['predicted_label']=pred;predictions['cluster']=routes
    predictions.to_csv(folder/'predictions.csv',index=False)
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(test.traffic_label,pred,labels=CLASSES,normalize='true',values_format='.2f',ax=ax,colorbar=False)
    ax.set_title(f'{name}: {train_level} → {test_level}');fig.tight_layout();fig.savefig(folder/'confusion.png',dpi=150);plt.show()
    manifest=dict(scenario=name,train_level=train_level,test_level=test_level,seed=SEED,window=WINDOW,ddof=0,
       mbk=MBK_CONFIG,rf=RF_CONFIG,expert_numeric=model['nums'],expert_categorical=model['cats'],
       encoded_features=model['encoded_features'],train_records=len(model['train']),test_records=len(test),
       train_silhouette=model['silhouette'],files=audit,sklearn=sklearn.__version__,
       protocol='Exploratory chronological 10% mixed-level adaptation; tuple-purged shared tests; 40-tree experts.')
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Inference label-independence and streaming equivalence passed. Reports:',folder)
    return model,metrics


In [ ]:
def tuple_key(r):
    a,b=sorted([(str(r.SrcAddr),str(r.Sport)),(str(r.DstAddr),str(r.Dport))])
    return (str(r.Proto),a,b)
low=data[data.congestion_level.eq('Low')].copy()
added=[];candidates=[];split_rows=[]
for (label,level),g in data[~data.congestion_level.eq('Low')].groupby(['traffic_label','congestion_level'],sort=True):
    g=g.sort_values(['timestamp','source_row'],kind='stable')
    n=int(.10*len(g));added.append(g.iloc[:n].copy());candidates.append(g.iloc[n:].copy())
    split_rows.append(dict(label=label,level=level,total=len(g),added_to_train=n,candidate_test=len(g)-n))
mixed=pd.concat([low,*added],ignore_index=True)
train_keys={tuple_key(r) for r in mixed.itertuples()}
raw_test=pd.concat(candidates,ignore_index=True)
mask=np.array([tuple_key(r) not in train_keys for r in raw_test.itertuples()])
tests=raw_test.loc[mask].copy()
assert not train_keys.intersection({tuple_key(r) for r in tests.itertuples()})
splits=pd.DataFrame(split_rows)
counts=tests.groupby(['traffic_label','congestion_level']).size()
splits['retained_test']=[int(counts.get((r.label,r.level),0)) for r in splits.itertuples()]
splits['tuple_purged']=splits.candidate_test-splits.retained_test
assert (splits.retained_test>WINDOW).all()
display(splits);splits.to_csv(OUT/'split_audit.csv',index=False)
mixed[['source_file','source_row','traffic_label','congestion_level']].to_csv(OUT/'train_membership.csv',index=False)
tests[['source_file','source_row','traffic_label','congestion_level']].to_csv(OUT/'test_membership.csv',index=False)
print('Low records:',len(low),'Mixed records:',len(mixed),'Purged test records:',int((~mask).sum()))


In [ ]:
mixed_model, mixed_medium = run_mixed('mixed_medium',mixed,tests[tests.congestion_level.eq('Medium')].copy())
mixed_model, mixed_high = run_mixed('mixed_high',mixed,tests[tests.congestion_level.eq('High')].copy(),mixed_model)
routes=mixed_model['router'].predict(mixed_model['scaler'].transform(mixed_model['train'][TREND_COLUMNS]))
cluster_levels=pd.crosstab(pd.Series(routes,name='cluster'),mixed_model['train'].congestion_level)
display(cluster_levels);cluster_levels.to_csv(OUT/'train_cluster_levels.csv')


In [ ]:
low_model, low_medium = run_mixed('control_medium',low,tests[tests.congestion_level.eq('Medium')].copy())
low_model, low_high = run_mixed('control_high',low,tests[tests.congestion_level.eq('High')].copy(),low_model)
comparison=pd.concat({'mixed_medium':mixed_medium,'mixed_high':mixed_high,'control_medium':low_medium,'control_high':low_high},names=['experiment','method'])
display(comparison);comparison.to_csv(OUT/'comparison.csv')


## Executed results (seed 42)
All eight code cells completed, including inference label-independence and streaming/batch equivalence checks.

Accuracy on identical purged test records:

| Training | Test | CDR-MLC (40 trees/expert) | RF100 expert inputs | RF100 with raw timing |
|---|---|---:|---:|---:|
| Low only | Medium | 0.870304 | 0.892720 | 0.876611 |
| Low + 10% Medium/High | Medium | 0.890458 | 0.901083 | 0.898135 |
| Low only | High | 0.825544 | 0.875049 | 0.839955 |
| Low + 10% Medium/High | High | 0.850862 | 0.880814 | 0.865555 |

Mixed training improves CDR-MLC by 2.015 and 2.532 percentage points respectively, but pooled RF remains stronger. Both router and experts receive added training data, so these results do not isolate a router-only benefit. Training clusters still mix true congestion levels; cluster IDs are arbitrary.

The conservative global tuple purge removes many HTTP candidates (8,039 Medium and 6,338 High). Tuple reuse across independent captures is not necessarily duplicate traffic, so this protection changes the evaluation distribution. Report the split audit and compare only the controls on these same test records. Do not compare these absolute scores directly with earlier full-test results. This is a within-capture exploratory adaptation experiment and needs independent captures for a stronger claim.
